# Ekstraksi Fitur Time Series Polutan Udara dengan TSFEL

## Data Polutan Kabupaten Gresik

Notebook ini digunakan untuk melakukan ekstraksi fitur time series dari data kualitas udara harian di wilayah **Gresik** yang terdiri dari tiga jenis polutan, yaitu `NO2`, `SO2`, dan `CO`. Proses ekstraksi fitur dilakukan menggunakan library **TSFEL (Time Series Feature Extraction Library)**.

Dalam pembagian tugas pada satu kelas, bagian yang dikerjakan penulis berfokus pada penjelasan serta pembuktian secara manual terhadap dua fitur TSFEL, yaitu `ecdf_slope` dan `entropy`. Notebook ini juga mencakup tahapan pembersihan data dan proses ekstraksi sebanyak 68 fitur TSFEL dari masing-masing polutan yang nantinya digunakan sebagai data untuk proses clustering.

Secara umum, notebook ini terdiri dari lima bagian utama:

1. Penjelasan mengenai fitur `ecdf_slope` dan `entropy`, termasuk proses perhitungan manual serta perbandingannya dengan hasil yang diperoleh dari TSFEL.
2. Data preparation yang mencakup pemeriksaan outlier dan penanganan missing value pada data `NO2`, `SO2`, dan `CO`.
3. Proses ekstraksi 68 fitur TSFEL dari masing-masing data polutan.
4. Penyajian hasil ekstraksi fitur dalam bentuk tabel dan visualisasi.

In [1]:
!pip install tsfel -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 39.1 MB/s eta 0:00:00


**Penjelasan kode:** `!pip install tsfel -q` digunakan untuk memasang library TSFEL pada Google Colab agar dapat digunakan untuk proses feature extraction data polutan.

## 1. Penjelasan Fitur `ecdf_slope` dan `entropy`

Bagian ini membahas dua fitur yang menjadi tanggung jawab penulis dalam pembagian tugas satu kelas: `ecdf_slope` dan `entropy`, keduanya termasuk dalam kelompok *distribution and complexity features*. Sebagai gambaran umum, berikut ringkasan pengelompokan seluruh 68 fitur yang dihasilkan TSFEL, sebelum masuk ke pembahasan mendalam kedua fitur tersebut.

### 1.1 Ringkasan Pengelompokan 68 Fitur TSFEL

Fitur yang dihasilkan menggunakan TSFEL terdiri dari beberapa kelompok berdasarkan karakteristik data yang dianalisis. Setiap kelompok memiliki fungsi yang berbeda, seperti menggambarkan nilai statistik, perubahan data, karakteristik frekuensi, kompleksitas sinyal, hingga distribusi data. Pengelompokan 68 fitur yang digunakan dapat dilihat pada tabel berikut.

#### A. Statistical Features

| No. | Fitur | Penjelasan |
|---|---|---|
| 1 | `calc_max` | Menentukan nilai paling tinggi atau maksimum dari data. |
| 2 | `calc_mean` | Menghitung nilai rata-rata untuk mengetahui nilai umum dari data. |
| 3 | `calc_median` | Menentukan nilai tengah setelah data diurutkan. |
| 4 | `calc_min` | Menentukan nilai paling rendah atau minimum dari data. |
| 5 | `calc_std` | Mengukur seberapa jauh nilai data menyebar dari nilai rata-ratanya. |
| 6 | `calc_var` | Mengukur tingkat variasi atau penyebaran nilai dalam data. |
| 7 | `kurtosis` | Mengukur tingkat keruncingan atau bentuk distribusi data. |
| 8 | `skewness` | Mengukur kemiringan distribusi data ke arah kanan atau kiri. |
| 9 | `interq_range` | Mengukur jarak antara kuartil pertama dan kuartil ketiga untuk melihat penyebaran bagian tengah data. |
| 10 | `mean_abs_deviation` | Mengukur rata-rata jarak absolut setiap nilai terhadap rata-rata data. |
| 11 | `median_abs_deviation` | Mengukur penyimpangan nilai data berdasarkan median. |
| 12 | `hist_mode` | Menentukan nilai yang paling sering muncul dalam data. |
| 13 | `rms` | Mengukur besarnya sinyal berdasarkan akar dari rata-rata kuadrat nilai data. |
| 14 | `mse` | Mengukur rata-rata kuadrat perbedaan atau kesalahan antar nilai data. |

#### B. Temporal Features

| No. | Fitur | Penjelasan |
|---|---|---|
| 1 | `autocorr` | Mengukur hubungan antara suatu nilai dengan nilai sebelumnya dalam urutan data. |
| 2 | `mean_abs_diff` | Menghitung rata-rata perubahan absolut antara nilai yang berurutan. |
| 3 | `mean_diff` | Menghitung rata-rata perubahan antara nilai yang berurutan. |
| 4 | `median_abs_diff` | Menentukan nilai tengah dari perubahan absolut antar nilai. |
| 5 | `median_diff` | Menentukan nilai tengah dari perubahan antar nilai. |
| 6 | `sum_abs_diff` | Menghitung total perubahan absolut pada seluruh data. |
| 7 | `negative_turning` | Menghitung perubahan arah sinyal yang menunjukkan pola penurunan. |
| 8 | `positive_turning` | Menghitung perubahan arah sinyal yang menunjukkan pola kenaikan. |
| 9 | `zero_cross` | Menghitung berapa kali sinyal melewati atau berubah tanda terhadap nilai nol. |
| 10 | `slope` | Mengukur kecenderungan data untuk naik atau turun. |
| 11 | `distance` | Mengukur jarak atau perubahan antara nilai-nilai pada sinyal. |
| 12 | `neighbourhood_peaks` | Mengidentifikasi jumlah puncak lokal yang terdapat pada sinyal. |
| 13 | `pk_pk_distance` | Mengukur jarak antara nilai puncak dan nilai lembah pada sinyal. |

#### C. Spectral Features

| No. | Fitur | Penjelasan |
|---|---|---|
| 1 | `fundamental_frequency` | Menentukan frekuensi dasar yang menjadi komponen utama pada sinyal. |
| 2 | `max_frequency` | Menentukan frekuensi yang memiliki komponen atau kekuatan paling tinggi. |
| 3 | `max_power_spectrum` | Menentukan nilai daya spektrum yang paling besar. |
| 4 | `median_frequency` | Menentukan frekuensi yang membagi energi spektrum menjadi dua bagian yang sama. |
| 5 | `power_bandwidth` | Mengukur rentang frekuensi yang mencakup sebagian besar energi sinyal. |
| 6 | `spectral_centroid` | Menentukan titik pusat distribusi energi pada spektrum frekuensi. |
| 7 | `spectral_decrease` | Mengukur tingkat penurunan energi pada spektrum dari frekuensi rendah ke tinggi. |
| 8 | `spectral_distance` | Mengukur perbedaan karakteristik atau distribusi energi pada spektrum. |
| 9 | `spectral_entropy` | Mengukur tingkat ketidakaturan atau penyebaran energi pada spektrum frekuensi. |
| 10 | `spectral_kurtosis` | Mengukur tingkat keruncingan distribusi energi pada spektrum. |
| 11 | `spectral_positive_turning` | Menghitung perubahan arah positif pada spektrum. |
| 12 | `spectral_roll_off` | Menentukan batas frekuensi yang mencakup sebagian besar energi spektrum. |
| 13 | `spectral_roll_on` | Menentukan bagian awal ketika energi pada spektrum mulai meningkat. |
| 14 | `spectral_skewness` | Mengukur kemiringan distribusi energi pada spektrum. |
| 15 | `spectral_slope` | Mengukur kecenderungan spektrum untuk meningkat atau menurun. |
| 16 | `spectral_spread` | Mengukur seberapa luas energi tersebar di sekitar pusat spektrum. |
| 17 | `spectral_variation` | Mengukur perubahan karakteristik energi pada spektrum. |
| 18 | `spectrogram_mean_coeff` | Menghitung nilai rata-rata koefisien yang terdapat pada spectrogram. |

#### D. Fractal Features

| No. | Fitur | Penjelasan |
|---|---|---|
| 1 | `dfa` | Mengukur pola fluktuasi jangka panjang pada data. |
| 2 | `higuchi_fractal_dimension` | Mengukur tingkat kompleksitas bentuk atau pola pada sinyal. |
| 3 | `hurst_exponent` | Menggambarkan kecenderungan data untuk mempertahankan atau mengubah pola sebelumnya. |
| 4 | `maximum_fractal_length` | Mengukur panjang maksimum pola fraktal yang terdapat pada sinyal. |
| 5 | `petrosian_fractal_dimension` | Mengukur kompleksitas sinyal berdasarkan perubahan arah pada data. |

#### E. Cepstral Features

| No. | Fitur | Penjelasan |
|---|---|---|
| 1 | `lpcc` | Menghasilkan karakteristik sinyal menggunakan Linear Prediction Cepstral Coefficients (LPCC). |
| 2 | `mfcc` | Menghasilkan karakteristik sinyal menggunakan Mel-Frequency Cepstral Coefficients (MFCC). |

#### F. Wavelet Features

| No. | Fitur | Penjelasan |
|---|---|---|
| 1 | `wavelet_abs_mean` | Menghitung rata-rata nilai absolut dari koefisien wavelet. |
| 2 | `wavelet_energy` | Mengukur energi sinyal berdasarkan koefisien wavelet. |
| 3 | `wavelet_entropy` | Mengukur tingkat ketidakaturan distribusi energi pada koefisien wavelet. |
| 4 | `wavelet_std` | Mengukur penyebaran nilai koefisien wavelet. |
| 5 | `wavelet_var` | Mengukur tingkat variasi nilai koefisien wavelet. |

#### G. Distribution and Complexity Features

| No. | Fitur | Penjelasan |
|---|---|---|
| 1 | `abs_energy` | Mengukur total energi data berdasarkan jumlah kuadrat nilai sinyal. |
| 2 | `average_power` | Mengukur rata-rata kekuatan atau energi yang terdapat pada sinyal. |
| 3 | `auc` | Menghitung luas area di bawah kurva data. |
| 4 | `calc_centroid` | Menentukan titik pusat atau lokasi rata-rata distribusi data. |
| 5 | `ecdf` | Menggambarkan distribusi kumulatif dari nilai-nilai pada data. |
| 6 | `ecdf_percentile` | Menentukan posisi atau nilai data berdasarkan persentil tertentu. |
| 7 | `ecdf_percentile_count` | Menghitung jumlah data yang berada pada persentil tertentu. |
| 8 | `ecdf_slope` | Mengukur kemiringan distribusi kumulatif berdasarkan perubahan persentil data. |
| 9 | `entropy` | Mengukur tingkat ketidakpastian atau keragaman nilai yang terdapat dalam data. |
| 10 | `human_range_energy` | Mengukur energi sinyal pada rentang frekuensi tertentu yang berkaitan dengan rentang pendengaran manusia. |
| 11 | `lempel_ziv` | Mengukur tingkat kompleksitas pola berdasarkan kemunculan pola-pola dalam sinyal. |

### 1.2 Import Library dan Menyiapkan Sinyal

Dataset berisi data harian. Kolom `date` digunakan sebagai indeks waktu, sedangkan nilai polutan digunakan sebagai sinyal untuk perhitungan `ecdf_slope` dan `entropy`.


In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LOKASI_DATA = "polutan_gresik_timeseries.csv"

df = pd.read_csv(LOKASI_DATA)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

polutan = ["NO2", "SO2", "CO"]

print("Ukuran data:", df.shape)

display(df.head())

Ukuran data: (366, 7)


,date,CO,NO2,HCHO,O3,SO2,CH4
0,2025-08-31,0.027550,0.000024,0.000145,0.116252,0.000210,NaN
1,2025-09-01,0.023488,0.000045,0.000042,0.114932,0.000058,NaN
2,2025-09-02,0.025977,0.000028,0.000089,0.115227,0.000534,NaN
3,2025-09-03,0.026092,0.000033,0.000225,0.113764,-0.000142,NaN
4,2025-09-04,0.029989,0.000060,0.000133,0.113487,0.000649,NaN


**Penjelasan kode:** Kode ini digunakan untuk mengimpor library yang diperlukan, membaca dataset `polutan_gresik_timeseries.csv`, mengubah kolom `date` menjadi format tanggal, mengurutkan data berdasarkan tanggal, serta menentukan tiga polutan yang akan dianalisis, yaitu `NO2`, `SO2`, dan `CO`. Ukuran dan beberapa baris awal dataset kemudian ditampilkan untuk memeriksa data.

In [88]:
# Memeriksa missing value dan menyiapkan sinyal
print("Missing value:")
display(df[polutan].isna().sum())

signals = {}

for col in polutan:
    df[col] = pd.to_numeric(df[col], errors="coerce")

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    df.loc[
        (df[col] < batas_bawah) | (df[col] > batas_atas),
        col
    ] = np.nan

    data_bersih = (
        df.set_index("date")[col]
        .interpolate(method="time")
        .ffill()
        .bfill()
    )

    signals[col] = data_bersih.astype(float).to_numpy()

for col, signal in signals.items():
    print(
        f"{col}: n={len(signal)}, "
        f"min={signal.min():.12g}, max={signal.max():.12g}"
    )

Missing value:


,0
NO2,195
SO2,151
CO,166


NO2: n=366, min=8.48615491122e-06, max=0.000110585489892
SO2: n=366, min=-0.000616634613834, max=0.000835011538584
CO: n=366, min=0.0210837982595, max=0.0376965887845


**Penjelasan kode:** Kode ini digunakan untuk memeriksa *missing value*, mengubah data polutan menjadi numerik, mendeteksi *outlier* menggunakan metode IQR, lalu mengganti nilai tersebut dengan `NaN`. Nilai kosong kemudian diisi menggunakan interpolasi berdasarkan waktu, dilanjutkan dengan `ffill()` dan `bfill()`. Hasil akhirnya disimpan sebagai sinyal numerik untuk perhitungan fitur TSFEL.

### 1.3 `ecdf_slope(signal[, p_init, p_end])`

#### Deskripsi

`ecdf_slope` menghitung **kemiringan ECDF (*Empirical Cumulative Distribution Function*)** antara dua probabilitas atau persentil.

ECDF menunjukkan proporsi data yang memiliki nilai kurang dari atau sama dengan suatu nilai tertentu.

Untuk data yang telah diurutkan:

$$
x_{(1)} \leq x_{(2)} \leq \cdots \leq x_{(n)}
$$

TSFEL membentuk nilai ECDF:

$$
y_i = \frac{i}{n}, \quad i=1,2,\ldots,n
$$

Kemudian untuk `p_init` dan `p_end`, TSFEL mengambil:

$$
x_{\text{init}} = \max\{x_i : y_i \leq p_{\text{init}}\}
$$

$$
x_{\text{end}} = \max\{x_i : y_i \leq p_{\text{end}}\}
$$

Selanjutnya, kemiringan ECDF dihitung dengan:

$$
\boxed{
ecdf\_slope =
\frac{p_{\text{end}}-p_{\text{init}}}
{x_{\text{end}}-x_{\text{init}}}
}
$$

#### Parameter yang digunakan

TSFEL memiliki nilai default:

```python
p_init = 0.5
p_end  = 0.75
```

Jadi, notebook ini menggunakan **`p_init = 0.5` dan `p_end = 0.75`**, bukan 0.25 dan 0.75.

#### Catatan penting

TSFEL tidak menggunakan `np.percentile()` untuk menghitung `ecdf_slope`. Oleh karena itu, perhitungan manual pada notebook ini juga **tidak menggunakan `np.percentile()`**.

Perhitungan manual mengikuti langkah yang digunakan TSFEL, yaitu:

1. Mengurutkan data dari nilai terkecil hingga terbesar.
2. Membentuk ECDF dengan rumus $y_i=i/n$.
3. Menentukan $x_{\text{init}}$ berdasarkan $y_i \leq p_{\text{init}}$.
4. Menentukan $x_{\text{end}}$ berdasarkan $y_i \leq p_{\text{end}}$.
5. Menghitung `ecdf_slope` menggunakan rumus di atas.
6. Membandingkan hasil perhitungan manual dengan hasil `ecdf_slope` dari TSFEL.


In [64]:
from tsfel.feature_extraction.features import ecdf_slope, entropy

**Penjelasan kode:** `from tsfel.feature_extraction.features import ecdf_slope, entropy` digunakan untuk mengambil fungsi `ecdf_slope` dan `entropy` dari TSFEL agar dapat digunakan dalam perhitungan fitur pada data time series.

#### 1.3.1 Perhitungan Manual `ecdf_slope`

Fungsi berikut dibuat untuk meniru langkah TSFEL:

1. Urutkan data.
2. Bentuk ECDF dengan `1/n, 2/n, ..., n/n`.
3. Ambil nilai `x` terbesar dengan ECDF `<= p_init`.
4. Ambil nilai `x` terbesar dengan ECDF `<= p_end`.
5. Hitung slope.

Ini adalah bagian penting yang membuat hasil manual dapat sama dengan TSFEL.


In [89]:
def calc_ecdf_manual_tsfel(signal):
    x = np.sort(np.asarray(signal, dtype=float))
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y


def ecdf_slope_manual_tsfel(signal, p_init=0.5, p_end=0.75):
    x, y = calc_ecdf_manual_tsfel(signal)

    # Nilai x terbesar dengan ECDF <= p_init dan <= p_end
    x_init = x[y <= p_init].max()
    x_end = x[y <= p_end].max()

    # Rumus TSFEL
    slope = (p_end - p_init) / (x_end - x_init)

    return slope, x_init, x_end

**Penjelasan kode:** Fungsi pertama mengurutkan data sinyal dan menghitung nilai ECDF berdasarkan posisi data. Fungsi kedua menggunakan nilai ECDF pada probabilitas 0,50 dan 0,75 untuk menentukan `x_init` dan `x_end`, kemudian menghitung kemiringan ECDF dengan rumus yang digunakan TSFEL.

In [90]:
# Contoh manual untuk setiap polutan
ecdf_manual_results = []

for col, signal in signals.items():
    slope, x_init, x_end = ecdf_slope_manual_tsfel(
        signal,
        p_init=0.5,
        p_end=0.75
    )

    ecdf_manual_results.append({
        "Polutan": col,
        "Jumlah data": len(signal),
        "p_init": 0.5,
        "p_end": 0.75,
        "x_init": x_init,
        "x_end": x_end,
        "ECDF slope manual": slope
    })

ecdf_manual_df = pd.DataFrame(ecdf_manual_results)
display(ecdf_manual_df)


,Polutan,Jumlah data,p_init,p_end,x_init,x_end,ECDF slope manual
0,NO2,366,0.5,0.75,0.000059197691206,0.000069871275628,23422.309705949122872
1,SO2,366,0.5,0.75,0.000130338055896,0.000295632554399,1512.452031159700937
2,CO,366,0.5,0.75,0.028473624959588,0.030595061870722,117.844654577265800


**Penjelasan kode:** Kode ini menghitung nilai \`ecdf_slope\` secara manual untuk setiap polutan berdasarkan sinyal yang telah dibersihkan. Hasil perhitungan mencakup jumlah data, probabilitas awal dan akhir, nilai \`x_init\` dan \`x_end\`, serta nilai \`ECDF slope\`, kemudian ditampilkan dalam bentuk tabel.

#### 1.3.2 Pembuktian dengan TSFEL

Sekarang fungsi manual dibandingkan dengan fungsi resmi:

```python
tsfel.feature_extraction.features.ecdf_slope()
```

Karena algoritma manual mengikuti implementasi TSFEL, nilai keduanya seharusnya sama hingga perbedaan pembulatan floating-point.


In [91]:
ecdf_tsfel_results = []

for col, signal in signals.items():
    value = ecdf_slope(signal, p_init=0.5, p_end=0.75)

    ecdf_tsfel_results.append({
        "Polutan": col,
        "ECDF slope TSFEL": float(value)
    })

ecdf_tsfel_df = pd.DataFrame(ecdf_tsfel_results)

ecdf_comparison = ecdf_manual_df[
    ["Polutan", "ECDF slope manual"]
].merge(ecdf_tsfel_df, on="Polutan")

ecdf_comparison["Selisih"] = (
    ecdf_comparison["ECDF slope manual"]
    - ecdf_comparison["ECDF slope TSFEL"]
)

display(ecdf_comparison)


,Polutan,ECDF slope manual,ECDF slope TSFEL,Selisih
0,NO2,23422.309705949122872,23422.309705949122872,0.0
1,SO2,1512.452031159700937,1512.452031159700937,0.0
2,CO,117.844654577265800,117.844654577265800,0.0


**Penjelasan kode:** Kode ini menghitung nilai `ecdf_slope` menggunakan fungsi TSFEL untuk setiap polutan, kemudian menggabungkannya dengan hasil perhitungan manual. Kolom `Selisih` digunakan untuk melihat perbedaan antara hasil manual dan hasil dari TSFEL.

### 1.4 `entropy(signal[, prob])`

#### Deskripsi

`entropy` pada TSFEL menggunakan **Shannon entropy** untuk mengukur tingkat ketidakpastian atau keragaman distribusi nilai dalam suatu sinyal.

Untuk `prob="standard"`, TSFEL menghitung probabilitas berdasarkan frekuensi setiap nilai unik:

$$
p_i = \frac{c_i}{n}
$$

dengan:

* $c_i$ = jumlah kemunculan nilai ke-$i$
* $n$ = jumlah seluruh data

Shannon entropy dihitung dengan rumus:

$$
H = -\sum_i p_i \log_2(p_i)
$$

TSFEL kemudian **menormalisasi** nilai entropy tersebut dengan:

$$
\boxed{
H_{\text{norm}} =
\frac{-\sum_i p_i\log_2(p_i)}
{\log_2(n)}
}
$$

Dengan normalisasi tersebut, nilai entropy menjadi lebih mudah dibandingkan antar-sinyal dengan jumlah data yang berbeda.

Jadi, untuk membuktikan hasil `entropy` dari TSFEL, perhitungan manual harus menggunakan **rumus Shannon entropy yang kemudian dinormalisasi** seperti pada rumus di atas.

#### Parameter

Default TSFEL:

```python
prob = "standard"
```

Artinya, probabilitas diperoleh berdasarkan **frekuensi nilai unik**, bukan dengan membagi data ke dalam histogram 10-bin.

#### Langkah Perhitungan Manual

1. Tentukan seluruh nilai unik pada data.
2. Hitung frekuensi kemunculan setiap nilai unik ($c_i$).
3. Hitung probabilitas setiap nilai dengan:

$$
p_i = \frac{c_i}{n}
$$

4. Hitung Shannon entropy:

$$
H = -\sum_i p_i\log_2(p_i)
$$

5. Normalisasi entropy:

$$
H_{\text{norm}} =
\frac{H}{\log_2(n)}
$$

6. Bandingkan hasil perhitungan manual dengan hasil `entropy` dari TSFEL.


In [92]:
def entropy_manual_tsfel(signal):
    signal = np.asarray(signal, dtype=float)

    # Sama seperti np.unique(signal, return_counts=True)
    values, counts = np.unique(signal, return_counts=True)

    # Probabilitas masing-masing nilai
    p = counts / counts.sum()

    # Hilangkan probabilitas nol
    p_nonzero = p[p != 0]

    # Normalisasi entropy seperti TSFEL
    n = len(signal)

    if np.log2(n) == 1:
        return 0.0, values, counts, p

    raw_entropy = -np.sum(p_nonzero * np.log2(p_nonzero))
    normalized_entropy = raw_entropy / np.log2(n)

    return normalized_entropy, values, counts, p

**Penjelasan kode:** Fungsi `entropy_manual_tsfel()` digunakan untuk menghitung nilai `entropy` secara manual. Data dihitung berdasarkan frekuensi setiap nilai, kemudian diubah menjadi probabilitas dan digunakan untuk menghitung Shannon entropy yang selanjutnya dinormalisasi sesuai dengan perhitungan TSFEL.

In [93]:
# Contoh sederhana untuk memahami perhitungan entropy
contoh = np.array([1, 1, 1, 2, 2, 3])

H, values, counts, p = entropy_manual_tsfel(contoh)

contoh_table = pd.DataFrame({
    "Nilai": values,
    "Frekuensi": counts,
    "Probabilitas": p
})

display(contoh_table)

raw_H = -np.sum(p[p != 0] * np.log2(p[p != 0]))
normalized_H = raw_H / np.log2(len(contoh))

print("Raw Shannon entropy =", raw_H)
print("log2(n) =", np.log2(len(contoh)))
print("Normalized entropy =", normalized_H)

,Nilai,Frekuensi,Probabilitas
0,1.0,3,0.500000000000000
1,2.0,2,0.333333333333333
2,3.0,1,0.166666666666667


Raw Shannon entropy = 1.4591479170272448
log2(n) = 2.584962500721156
Normalized entropy = 0.5644754678724236


**Penjelasan kode:** Kode ini menggunakan data contoh `[1, 1, 1, 2, 2, 3]` untuk memperlihatkan proses perhitungan `entropy`. Frekuensi dan probabilitas setiap nilai ditampilkan dalam tabel, kemudian dihitung nilai Shannon entropy dan entropy yang telah dinormalisasi sesuai dengan perhitungan TSFEL.

#### 1.4.1 Perhitungan Manual Entropy untuk NO2, SO2, dan CO

Untuk setiap polutan, probabilitas dihitung berdasarkan jumlah kemunculan masing-masing nilai unik.

Jika seluruh nilai valid dalam suatu polutan memiliki nilai yang unik, maka setiap nilai hanya muncul satu kali. Dengan demikian:

$$
p_i = \frac{1}{n}
$$

Shannon entropy menjadi:

$$
H
=
-n\left(\frac{1}{n}\log_2\frac{1}{n}\right)
$$

sehingga:

$$
H = \log_2(n)
$$

Kemudian, TSFEL melakukan normalisasi entropy dengan membagi nilai entropy dengan $\log_2(n)$:

$$
H_{\text{norm}}
=
\frac{\log_2(n)}{\log_2(n)}
$$

sehingga:

$$
\boxed{H_{\text{norm}}=1}
$$

Dengan demikian, apabila seluruh nilai valid pada data **NO2, SO2, dan CO** bersifat unik, maka hasil `entropy` dari TSFEL akan bernilai:

$$
\boxed{entropy = 1.0}
$$

Nilai tersebut menunjukkan bahwa berdasarkan perhitungan entropy yang digunakan TSFEL, seluruh nilai dalam sinyal memiliki frekuensi kemunculan yang sama, yaitu masing-masing muncul satu kali.


In [94]:
entropy_manual_results = []
entropy_detail = {}

for col, signal in signals.items():
    H, values, counts, p = entropy_manual_tsfel(signal)

    entropy_manual_results.append({
        "Polutan": col,
        "Jumlah data": len(signal),
        "Jumlah nilai unik": len(values),
        "Entropy manual": H
    })

    entropy_detail[col] = pd.DataFrame({
        "Nilai": values,
        "Frekuensi": counts,
        "Probabilitas": p
    })

entropy_manual_df = pd.DataFrame(entropy_manual_results)
display(entropy_manual_df)

print("Contoh tabel probabilitas untuk NO2:")
display(entropy_detail["NO2"].head(10))

,Polutan,Jumlah data,Jumlah nilai unik,Entropy manual
0,NO2,366,366,1.0
1,SO2,366,366,1.0
2,CO,366,366,1.0


Contoh tabel probabilitas untuk NO2:


,Nilai,Frekuensi,Probabilitas
0,0.000008486154911,1,0.002732240437158
1,0.000020169327399,1,0.002732240437158
2,0.000020721830879,1,0.002732240437158
3,0.000022071355488,1,0.002732240437158
4,0.000023438198696,1,0.002732240437158
5,0.000023913456971,1,0.002732240437158
6,0.000024331446184,1,0.002732240437158
7,0.000025576335247,1,0.002732240437158
8,0.000026170286219,1,0.002732240437158
9,0.000026198122214,1,0.002732240437158


**Penjelasan kode:** Kode ini menghitung `entropy` secara manual untuk masing-masing polutan `NO2`, `SO2`, dan `CO`, kemudian menyimpan hasilnya dalam tabel. Selain itu, frekuensi dan probabilitas setiap nilai disimpan untuk melihat detail perhitungan, dengan contoh tabel probabilitas ditampilkan untuk `NO2`.

#### 1.4.2 Pembuktian dengan TSFEL

In [95]:
entropy_tsfel_results = []

for col, signal in signals.items():
    value = entropy(signal, prob="standard")

    entropy_tsfel_results.append({
        "Polutan": col,
        "Entropy TSFEL": float(value)
    })

entropy_tsfel_df = pd.DataFrame(entropy_tsfel_results)

entropy_comparison = entropy_manual_df[
    ["Polutan", "Entropy manual"]
].merge(entropy_tsfel_df, on="Polutan")

entropy_comparison["Selisih"] = (
    entropy_comparison["Entropy manual"]
    - entropy_comparison["Entropy TSFEL"]
)

display(entropy_comparison)

,Polutan,Entropy manual,Entropy TSFEL,Selisih
0,NO2,1.0,1.0,0.0
1,SO2,1.0,1.0,0.0
2,CO,1.0,1.0,0.0


**Penjelasan kode:** Kode ini menghitung nilai `entropy` menggunakan fungsi TSFEL untuk setiap polutan. Hasilnya kemudian dibandingkan dengan perhitungan manual sebelumnya, sedangkan kolom `Selisih` menunjukkan perbedaan antara kedua hasil tersebut.

### 1.5 Tabel dan Verifikasi Hasil pada Data Riil

Tabel berikut menggabungkan hasil manual dan hasil TSFEL untuk kedua fitur, dilanjutkan dengan pengecekan otomatis bahwa selisih keduanya mendekati nol.

In [96]:
final_comparison = (
    ecdf_comparison[["Polutan", "ECDF slope manual", "ECDF slope TSFEL", "Selisih"]]
    .rename(columns={"Selisih": "Selisih ECDF"})
    .merge(
        entropy_comparison[
            ["Polutan", "Entropy manual", "Entropy TSFEL", "Selisih"]
        ].rename(columns={"Selisih": "Selisih Entropy"}),
        on="Polutan"
    )
)

display(final_comparison)

print("\nPembulatan 6 angka:")
display(final_comparison.round(6))

,Polutan,ECDF slope manual,ECDF slope TSFEL,Selisih ECDF,Entropy manual,Entropy TSFEL,Selisih Entropy
0,NO2,23422.309705949122872,23422.309705949122872,0.0,1.0,1.0,0.0
1,SO2,1512.452031159700937,1512.452031159700937,0.0,1.0,1.0,0.0
2,CO,117.844654577265800,117.844654577265800,0.0,1.0,1.0,0.0



Pembulatan 6 angka:


,Polutan,ECDF slope manual,ECDF slope TSFEL,Selisih ECDF,Entropy manual,Entropy TSFEL,Selisih Entropy
0,NO2,23422.309706000000006,23422.309706000000006,0.0,1.0,1.0,0.0
1,SO2,1512.452031000000034,1512.452031000000034,0.0,1.0,1.0,0.0
2,CO,117.844655000000003,117.844655000000003,0.0,1.0,1.0,0.0


**Penjelasan kode:** Kode ini menggabungkan hasil perhitungan manual dan hasil TSFEL untuk fitur `ecdf_slope` dan `entropy` ke dalam satu tabel. Kolom `Selisih` digunakan untuk melihat perbedaan hasil manual dengan TSFEL, sedangkan `round(6)` menampilkan hasil dengan pembulatan hingga 6 angka desimal.

### 1.6 Kesimpulan Bagian 1

#### `ecdf_slope`

`ecdf_slope` digunakan untuk mengukur kemiringan ECDF antara probabilitas 0,5 dan 0,75. Pada TSFEL, nilai ECDF dibentuk dari data yang telah diurutkan dengan probabilitas kumulatif berdasarkan $i/n$. Nilai kemiringan dihitung menggunakan rumus:

$$
\text{slope} = \frac{0.75 - 0.50}{x_{75} - x_{50}}
$$

#### `entropy`

`entropy` digunakan untuk mengukur tingkat ketidakpastian dalam distribusi nilai menggunakan Shannon entropy. Pada `prob="standard"`, probabilitas diperoleh berdasarkan frekuensi kemunculan setiap nilai unik. TSFEL kemudian menormalisasi Shannon entropy menggunakan $\log_2(n)$ sehingga nilai entropy berada pada skala normalisasi.

#### Hasil

Hasil perhitungan manual dan TSFEL memiliki nilai yang sama karena perhitungan manual pada notebook ini mengikuti mekanisme perhitungan yang digunakan oleh TSFEL.

**Catatan untuk laporan:** Jangan menggunakan `np.percentile()` atau entropy berbasis 10-bin jika tujuan perhitungan adalah membuktikan hasil fitur TSFEL secara langsung, karena metode tersebut dapat menghasilkan nilai yang berbeda dari implementasi TSFEL.

## 2. Data Preparation: Deteksi Outlier, Imputasi Missing Value dan Ekstraksi Fitur Polutan




### 2.1 Deteksi Outlier, Imputasi Missing Value dan Ekstraksi Fitur Polutan NO2

In [15]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
df = pd.read_csv('polutan_gresik_timeseries.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'NO2'

# --- FIX: paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN ---
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: {n_missing_before}")

Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- 2. Daftar PERSIS fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")
extracted_features_final.to_csv(f'{target_pollutant}_Gresik_TSFEL.csv', index=False)

Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: 178
Jumlah fitur yang diminta: 68
Berhasil! Jumlah fitur yang dihasilkan untuk NO2: 68


**Penjelasan kode:** Kode digunakan untuk membaca data NO2, mengubah format tanggal, serta memastikan nilai polutan dalam bentuk numerik. Data kemudian diperiksa untuk menemukan outlier menggunakan metode IQR dan nilai kosong diisi dengan interpolasi berdasarkan waktu. Setelah data bersih, TSFEL digunakan untuk menghitung 68 fitur yang telah ditentukan. Hasilnya disimpan dalam DataFrame dan diekspor ke file CSV.

In [16]:
import pandas as pd

df_tsfel = pd.read_csv('NO2_Gresik_TSFEL.csv')

display(df_tsfel)

,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,...,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,0.000002,0.022417,3.0,4.175288e-09,184.039735,0.000124,0.000061,0.00006,0.000008,0.00002,...,0.154967,0.685525,7.180608e-10,0.00438,0.000003,0.000031,2.145496,0.000031,1.040073e-09,0.0


**Penjelasan kode:** `pandas` digunakan untuk membaca dan mengolah data dalam bentuk tabel. Fungsi `pd.read_csv()` digunakan untuk membaca file `NO2_Gresik_TSFEL.csv` dan menyimpannya ke dalam variabel `df_tsfel`. Selanjutnya, `display()` digunakan untuk menampilkan data tersebut dalam bentuk tabel.

### 2.2 Deteksi Outlier, Imputasi Missing Value dan Ekstraksi Fitur Polutan CO

In [17]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
df = pd.read_csv('polutan_gresik_timeseries.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'CO'

# --- FIX: paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN ---
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: {n_missing_before}")

Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- 2. Daftar PERSIS fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")
extracted_features_final.to_csv(f'{target_pollutant}_Gresik_TSFEL.csv', index=False)

Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: 158
Jumlah fitur yang diminta: 68
Berhasil! Jumlah fitur yang dihasilkan untuk CO: 68


**Penjelasan kode:** Kode digunakan untuk membaca data CO, mengubah format tanggal, serta memastikan nilai polutan dalam bentuk numerik. Data kemudian diperiksa untuk menemukan outlier menggunakan metode IQR dan nilai kosong diisi dengan interpolasi berdasarkan waktu. Setelah data bersih, TSFEL digunakan untuk menghitung 68 fitur yang telah ditentukan. Hasilnya disimpan dalam DataFrame dan diekspor ke file CSV.

In [18]:
import pandas as pd

df_tsfel = pd.read_csv('CO_Gresik_TSFEL.csv')

display(df_tsfel)

,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,...,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,0.307969,10.527944,2.0,0.000844,182.065409,0.03859,0.028848,0.028483,0.019385,0.00304,...,0.13338,0.660595,0.000015,0.662317,0.001686,0.007452,2.137603,0.007233,0.000059,0.0


**Penjelasan kode:** `pandas` digunakan untuk membaca dan mengolah data dalam bentuk tabel. Fungsi `pd.read_csv()` digunakan untuk membaca file `CO_Gresik_TSFEL.csv` dan menyimpannya ke dalam variabel `df_tsfel`. Selanjutnya, `display()` digunakan untuk menampilkan data tersebut dalam bentuk tabel.

### 2.3 Deteksi Outlier, Imputasi Missing Value dan Ekstraksi Fitur Polutan SO2

In [19]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
df = pd.read_csv('polutan_gresik_timeseries.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'SO2'

# --- FIX: paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN ---
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: {n_missing_before}")

Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- 2. Daftar PERSIS fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")
extracted_features_final.to_csv(f'{target_pollutant}_Gresik_TSFEL.csv', index=False)

Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: 147
Jumlah fitur yang diminta: 68
Berhasil! Jumlah fitur yang dihasilkan untuk SO2: 68


**Penjelasan kode:** Kode digunakan untuk membaca data SO2, mengubah format tanggal, serta memastikan nilai polutan dalam bentuk numerik. Data kemudian diperiksa untuk menemukan outlier menggunakan metode IQR dan nilai kosong diisi dengan interpolasi berdasarkan waktu. Setelah data bersih, TSFEL digunakan untuk menghitung 68 fitur yang telah ditentukan. Hasilnya disimpan dalam DataFrame dan diekspor ke file CSV.

In [20]:
import pandas as pd

df_tsfel = pd.read_csv('SO2_Gresik_TSFEL.csv')

display(df_tsfel)

,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,...,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,0.00003,0.072986,2.0,8.341181e-08,198.745858,0.000835,0.000142,0.000131,-0.000617,0.000251,...,0.154393,0.332211,1.054722e-07,0.069161,0.000009,0.000352,2.165275,0.000352,1.302620e-07,99.0


**Penjelasan kode:** `pandas` digunakan untuk membaca dan mengolah data dalam bentuk tabel. Fungsi `pd.read_csv()` digunakan untuk membaca file `SO2_Gresik_TSFEL.csv` dan menyimpannya ke dalam variabel `df_tsfel`. Selanjutnya, `display()` digunakan untuk menampilkan data tersebut dalam bentuk tabel.